# Motor Exercise 3 — looking for the onset of sustained motion

Plot repeated wheel-speed measurements against requested PWM and use the visible transition to plan a finer deadband test.

Start with the supplied synthetic example so that every cell runs before you have collected data. The example demonstrates the plotting route; it is not evidence about your robot and is not a result you should expect to reproduce. When you are ready, change only the settings in **Use the example or your own data** and run the notebook again.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
rng = np.random.default_rng(2026)


## 1. Use the example or your own data

Leave `USE_EXAMPLE_DATA` set to `True` on your first run. To use your measurements, upload the CSV, set it to `False`, and enter the filename. This is the main cell you need to edit.

Expected CSV columns: `wheel`, `direction`, `PWM`, `trial_num`, `sample_num`, and `speed_cps`.

Use consistent labels such as `left`/`right` and `forward`/`reverse`.


In [ ]:
USE_EXAMPLE_DATA = True
CSV_FILENAME = "motor_exercise03_deadband.csv"

print("Using:", "synthetic example" if USE_EXAMPLE_DATA else CSV_FILENAME)


## 2. Create the small synthetic example

The generated transition differs between wheels and directions. These values illustrate the plot only; they are not expected thresholds.


In [ ]:
example_rows = []
example_thresholds = {
    ("left", "forward"): 42,
    ("left", "reverse"): 48,
    ("right", "forward"): 38,
    ("right", "reverse"): 45,
}

for (wheel, direction), threshold in example_thresholds.items():
    sign = 1 if direction == "forward" else -1
    for pwm_magnitude in range(20, 71, 5):
        for trial_num in range(1, 5):
            for sample_num in range(8):
                typical_speed = max(pwm_magnitude - threshold, 0) * 11
                example_rows.append({
                    "wheel": wheel,
                    "direction": direction,
                    "PWM": sign * pwm_magnitude,
                    "trial_num": trial_num,
                    "sample_num": sample_num,
                    "speed_cps": sign * (typical_speed + rng.normal(0, 4)),
                })

example_data = pd.DataFrame(example_rows)


## 3. Load and preview the selected data

This is where your uploaded CSV enters the notebook. Check the first rows before continuing: column names, units and labels should match the exercise.


In [ ]:
if USE_EXAMPLE_DATA:
    data = example_data.copy()
else:
    data = pd.read_csv(CSV_FILENAME)

data.head()


## 4. Choose one wheel and direction

Begin with one case so the graph remains readable. Change these two labels and rerun the following cells to inspect the others.


In [ ]:
SELECTED_WHEEL = "left"
SELECTED_DIRECTION = "forward"

selected = data.loc[
    (data["wheel"] == SELECTED_WHEEL)
    & (data["direction"] == SELECTED_DIRECTION)
].copy()

selected.head()


## 5. Plot every recorded speed

Each point is one speed observation. Keeping the repeated points visible helps distinguish a reliable start from a twitch or noisy estimate.


In [ ]:
sns.stripplot(
    data=selected,
    x="PWM",
    y="speed_cps",
    jitter=0.18,
    alpha=0.55,
    native_scale=True,
)
plt.axhline(0, color="black", linewidth=1)
plt.title(f"Repeated speed observations: {SELECTED_WHEEL}, {SELECTED_DIRECTION}")
plt.xlabel("Requested PWM")
plt.ylabel("Encoder speed (counts/s)")
plt.show()


## 6. Summarise each requested PWM

`groupby(...)` gathers rows with the same PWM. The median shows the typical result without hiding the raw points above.


In [ ]:
command_summary = (
    selected.groupby("PWM", as_index=False)
    .agg(
        typical_speed_cps=("speed_cps", "median"),
        repeat_spread_cps=("speed_cps", "std"),
    )
    .sort_values("PWM")
)

command_summary


In [ ]:
sns.lineplot(
    data=command_summary,
    x="PWM",
    y="typical_speed_cps",
    marker="o",
)
plt.axhline(0, color="black", linewidth=1)
plt.title("Typical speed across the tested PWM values")
plt.xlabel("Requested PWM")
plt.ylabel("Median encoder speed (counts/s)")
plt.show()


## What to notice

- What is the largest tested command with no repeatable sustained motion?
- What is the smallest tested command with clear repeated motion?
- Which additional PWM values would narrow the interval efficiently?
- Repeat the view for both wheels and both directions before generalising.
